In [1]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import sys
import subprocess

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj.to_string(index=False) if hasattr(obj, "to_string") else obj)
    def Markdown(text):
        return text

# ---------------------------------------------------------------------
# 0. Ensure python-docx is available (auto-install if missing)
# ---------------------------------------------------------------------
def _ensure(pkg_import, pip_name=None):
    try:
        __import__(pkg_import)
        return True
    except ImportError:
        pip_name = pip_name or pkg_import
        print(f"[setup] Installing missing package: {pip_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pip_name])
            __import__(pkg_import)
            return True
        except Exception as e:
            print(f"[setup] Could not install {pip_name}: {e}")
            return False

DOCX_AVAILABLE = _ensure("docx", "python-docx")
try:
    from docx import Document
    from docx.shared import Pt
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    DOCX_AVAILABLE = False
    if not DOCX_AVAILABLE:
        warnings.warn("python-docx unavailable. Word report will be skipped.")

# ---------------------------------------------------------------------
# 1. File and column settings
# ---------------------------------------------------------------------
WORKBOOK_PATH = Path(r"C:\Users\rs2672\Desktop\AI Open\FinalWorkbook.xlsx")
if not WORKBOOK_PATH.exists():
    candidates = [
        Path.cwd() / "FinalWorkbook.xlsx",
        Path("/workspace/scratch/217783d20c9b/upload/FinalWorkbook.xlsx"),
    ]
    WORKBOOK_PATH = next((p for p in candidates if p.exists()), WORKBOOK_PATH)
if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(f"Workbook not found: {WORKBOOK_PATH}. Edit WORKBOOK_PATH above.")

PREFERRED_SHEETS = ["Audit Data", "Audit_Data", "AuditData", "Audit", "Audit Data Sheet"]
EXPORT_TABLES = True
BOOTSTRAP_REPS = 10000
RANDOM_SEED = 20261003

CTS_BASES = {
    "code": ["Code present", "Source code present", "Source code available"],
    "weights": ["Weights present", "Model weights present", "Weights available"],
    "training_data": ["Training data present", "Training data disclosed", "Training data available"],
    "preprocessing": ["Preprocessing disclosed", "Preprocessing pipeline disclosed"],
    "evaluation": ["Evaluation protocol disclosed", "Evaluation disclosed"],
    "documentation": ["Documentation adequate", "Documentation disclosed", "Technical documentation"],
    "alignment_safety": ["Alignment/safety disclosed", "Alignment safety disclosed", "Safety disclosed"],
}
RS_KEYS = ["code", "weights", "training_data", "evaluation"]

TRAINING_DEPTH_BASES = {
    "source_spec": ["Source spec depth", "Training data source depth"],
    "preprocessing": ["Preprocessing depth", "Data preprocessing depth"],
    "data_license": ["Data license depth", "Training data license depth"],
    "language_domain": ["Language/domain depth", "Language domain depth"],
    "synthetic_organic": ["Synthetic/organic ratio depth", "Synthetic organic ratio depth"],
}
TDDI_BINARY_BASES = {
    "source_spec": ["TDDI source specification", "Data source specified"],
    "preprocessing": ["TDDI preprocessing", "Data preprocessing specified"],
    "data_license": ["TDDI licensing clarity", "Training data licensing specified"],
    "language_domain": ["TDDI language domain distribution", "Language domain distribution specified"],
    "synthetic_organic": ["TDDI synthetic organic composition", "Synthetic organic composition specified"],
}
DEPTH7_COLUMNS = None

# ---------------------------------------------------------------------
# 2. Read workbook and locate the data sheet
# ---------------------------------------------------------------------
excel = pd.ExcelFile(WORKBOOK_PATH, engine="openpyxl")
print("Workbook:", WORKBOOK_PATH)
print("Sheets:", excel.sheet_names)
sheet = next((s for s in PREFERRED_SHEETS if s in excel.sheet_names), None)
if sheet is None:
    audit_candidates = [s for s in excel.sheet_names if "audit" in s.lower()]
    sheet = audit_candidates[0] if audit_candidates else None
if sheet is None:
    raise ValueError("No audit-data sheet found. Set the correct sheet name in PREFERRED_SHEETS.")
raw = pd.read_excel(WORKBOOK_PATH, sheet_name=sheet, engine="openpyxl")
raw.columns = [str(c).strip() for c in raw.columns]

DATA_LABEL = "LLM TRANSPARENCY AUDIT — RESULTS 3.1 to 3.3"
display(Markdown(f"## **{DATA_LABEL}**\n\nSelected sheet: `{sheet}`. The notebook computes component scores, summary statistics, and threshold counts from the supplied audit data."))

# ---------------------------------------------------------------------
# 3. Column mapping
# ---------------------------------------------------------------------
def norm(text):
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())

def find_col(columns, aliases):
    normalized = {norm(c): c for c in columns}
    for alias in aliases:
        if norm(alias) in normalized:
            return normalized[norm(alias)]
    for alias in aliases:
        a = norm(alias)
        for c in columns:
            if norm(c).startswith(a):
                return c
    return None

id_col = find_col(raw.columns, ["List No.", "Model ID", "ID"])
model_col = find_col(raw.columns, ["Model name", "Model name as supplied", "Model"])
year_col = find_col(raw.columns, ["Release year", "Release year as supplied", "Year"])
category_col = find_col(raw.columns, ["Category", "Disclosure category"])
if model_col is None:
    raise ValueError(f"Model-name column not found. Available columns: {list(raw.columns)}")

cts_cols = {key: find_col(raw.columns, aliases) for key, aliases in CTS_BASES.items()}
missing_cts = [key for key, col in cts_cols.items() if col is None]
if missing_cts:
    raise ValueError(f"Missing CTS component columns for {missing_cts}. Available columns: {list(raw.columns)}")

# ---------- ROBUST BINARY CONVERTER ----------
# Handles: 0/1, yes/no, y/n, true/false, present/absent, checkmarks, crosses,
# Unicode check/cross variants, emoji, and blanks.
POSITIVE_TOKENS = {
    "1", "yes", "y", "true", "t", "present", "available", "disclosed",
    "✓", "✔", "☑", "\u2713", "\u2714", "\u2611", "checkmark",
    "check", "filled", "included", "reported",
}
NEGATIVE_TOKENS = {
    "0", "no", "n", "false", "f", "absent", "unavailable", "notdisclosed",
    "×", "✗", "✘", "☒", "\u2717", "\u2718", "\u2612", "x", "cross",
    "missing", "none", "notreported",
}
def as_binary(series):
    """Convert to 0/1; leave anything unrecognised as NaN (do not force to 1)."""
    # Start from numeric interpretation
    numeric = pd.to_numeric(series, errors="coerce")
    out = numeric.where(numeric.isin([0, 1]))
    # Text interpretation
    text = series.astype("string").str.strip().str.lower()
    text_compact = text.str.replace(r"[\s\-_/]+", "", regex=True)
    pos_mask = text_compact.isin({t.replace(" ", "") for t in POSITIVE_TOKENS})
    neg_mask = text_compact.isin({t.replace(" ", "") for t in NEGATIVE_TOKENS})
    out = out.mask(pos_mask, 1).mask(neg_mask, 0)
    return out.astype(float)

model = raw.copy()
model[model_col] = model[model_col].astype("string").str.strip()
model = model[model[model_col].notna() & (model[model_col] != "")].copy()
if id_col:
    model[id_col] = pd.to_numeric(model[id_col], errors="coerce")
    model = model.sort_values(id_col, kind="stable", na_position="last").reset_index(drop=True)
else:
    id_col = "_inventory_row"
    model[id_col] = np.arange(1, len(model) + 1)

# ---------- DEBUG: show unique raw values per CTS column ----------
print("\n" + "="*74)
print("DEBUG: unique raw values found in each CTS source column")
print("="*74)
for key, col in cts_cols.items():
    if col is None:
        print(f"  {key}: COLUMN NOT FOUND")
        continue
    uniq = model[col].dropna().astype(str).str.strip().unique()
    sample = ", ".join(repr(u) for u in uniq[:15])
    print(f"  {key} ({col}): {len(uniq)} unique -> [{sample}]")
print("="*74)

for key, col in cts_cols.items():
    model[f"a_{key}"] = as_binary(model[col])
score_cols = [f"a_{key}" for key in CTS_BASES]
score_coverage = model[score_cols].notna().sum(axis=1)
model["CTS"] = model[score_cols].mean(axis=1).where(score_coverage.eq(7))
model["CTS_components_observed"] = score_coverage
model["RS"] = model[[f"a_{key}" for key in RS_KEYS]].mean(axis=1).where(
    model[[f"a_{key}" for key in RS_KEYS]].notna().all(axis=1)
)

# ---------- DEBUG: distribution of binary values per component ----------
print("\nDEBUG: binary value distribution per CTS component (1 / 0 / NaN)")
for key in CTS_BASES:
    s = model[f"a_{key}"]
    n_one = int((s == 1).sum()); n_zero = int((s == 0).sum()); n_nan = int(s.isna().sum())
    print(f"  {key}: 1={n_one}, 0={n_zero}, NaN={n_nan}")

# ---------- WARNING if no variance ----------
if model["CTS"].dropna().nunique() <= 1:
    print("\n[WARNING] CTS has no variance across rows. This usually means the CTS columns")
    print("          do not contain a mix of yes/no values. Check the debug output above")
    print("          and confirm the correct sheet/columns are being read.")

if year_col:
    model["Release_year_numeric"] = pd.to_numeric(model[year_col], errors="coerce")
else:
    model["Release_year_numeric"] = np.nan
if category_col:
    model["Category_label"] = model[category_col].astype("string").str.strip()
else:
    model["Category_label"] = pd.NA

training_depth_cols = {key: find_col(raw.columns, aliases) for key, aliases in TRAINING_DEPTH_BASES.items()}
if all(training_depth_cols.values()):
    for key, col in training_depth_cols.items():
        model[f"depth5_{key}"] = pd.to_numeric(model[col], errors="coerce")
    d5 = [f"depth5_{key}" for key in training_depth_cols]
    model["Training_data_depth_mean_5field"] = model[d5].mean(axis=1).where(model[d5].notna().all(axis=1))
    model["Training_data_depth_deficiency_proxy_5field"] = 1 - model["Training_data_depth_mean_5field"]
else:
    d5 = []

tddi_binary_cols = {key: find_col(raw.columns, aliases) for key, aliases in TDDI_BINARY_BASES.items()}
if all(tddi_binary_cols.values()):
    for key, col in tddi_binary_cols.items():
        model[f"tddi_{key}"] = as_binary(model[col])
    tddi_cols = [f"tddi_{key}" for key in TDDI_BINARY_BASES]
    tddi_coverage = model[tddi_cols].notna().sum(axis=1)
    model["TDDI_raw_0_5"] = model[tddi_cols].sum(axis=1).where(tddi_coverage.eq(5))
    model["TDDI_normalized_0_1"] = model["TDDI_raw_0_5"] / 5
else:
    tddi_cols = []

if DEPTH7_COLUMNS is not None:
    if len(DEPTH7_COLUMNS) != 7 or not all(c in raw.columns for c in DEPTH7_COLUMNS):
        raise ValueError("DEPTH7_COLUMNS must contain seven exact column names present in the workbook.")
    for i, col in enumerate(DEPTH7_COLUMNS, 1):
        model[f"d_{i}"] = pd.to_numeric(model[col], errors="coerce")
    d7 = [f"d_{i}" for i in range(1, 8)]
    model["DDI_7component"] = (1 - model[d7].mean(axis=1)).where(model[d7].notna().all(axis=1))
else:
    d7 = []

print("\nRows retained as inventory records:", len(model))
print("Distinct nonempty model-name strings:", model[model_col].nunique())
print("Distinct nonmissing IDs:", model[id_col].nunique(dropna=True))
print("Rows with complete seven-component CTS:", int(model["CTS"].notna().sum()))
print("Rows with complete four-component RS:", int(model["RS"].notna().sum()))

# ---------------------------------------------------------------------
# 4. Summary statistics
# ---------------------------------------------------------------------
rng = np.random.default_rng(RANDOM_SEED)
def describe_score(series, name):
    x = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    if len(x) == 0:
        return {"Metric": name, "n": 0, "mean": np.nan, "median": np.nan, "sample_SD": np.nan,
                "min": np.nan, "Q1": np.nan, "Q3": np.nan, "max": np.nan,
                "mean_95CI_low": np.nan, "mean_95CI_high": np.nan}
    boot = np.mean(rng.choice(x, size=(BOOTSTRAP_REPS, len(x)), replace=True), axis=1) if len(x) > 1 else np.repeat(x[0], BOOTSTRAP_REPS)
    return {"Metric": name, "n": len(x), "mean": np.mean(x), "median": np.median(x),
            "sample_SD": np.std(x, ddof=1) if len(x) > 1 else np.nan,
            "min": np.min(x), "Q1": np.quantile(x, .25), "Q3": np.quantile(x, .75), "max": np.max(x),
            "mean_95CI_low": np.quantile(boot, .025), "mean_95CI_high": np.quantile(boot, .975)}

summary_rows = [describe_score(model["CTS"], "CTS"), describe_score(model["RS"], "RS")]
if "TDDI_normalized_0_1" in model:
    summary_rows.append(describe_score(model["TDDI_normalized_0_1"], "TDDI (five binary subdimensions, normalized)"))
if "DDI_7component" in model:
    summary_rows.append(describe_score(model["DDI_7component"], "DDI (7 component-depth fields)"))
if "Training_data_depth_mean_5field" in model:
    summary_rows.append(describe_score(model["Training_data_depth_mean_5field"], "Training-data mean depth (5-field proxy)"))
    summary_rows.append(describe_score(model["Training_data_depth_deficiency_proxy_5field"], "Training-data depth-deficiency proxy (5 fields)"))
score_summary = pd.DataFrame(summary_rows)
display(Markdown(f"## {DATA_LABEL}: overall score statistics"))
display(score_summary.round(4))

cts_valid = model["CTS"].dropna()
if len(cts_valid):
    thresholds = pd.DataFrame([
        {"CTS band": "Low (CTS <= 0.25)", "n": int((cts_valid <= .25).sum()), "percent of complete CTS": 100*(cts_valid <= .25).mean()},
        {"CTS band": "Intermediate (0.25 < CTS < 0.75)", "n": int(((cts_valid > .25) & (cts_valid < .75)).sum()), "percent of complete CTS": 100*((cts_valid > .25) & (cts_valid < .75)).mean()},
        {"CTS band": "High (CTS >= 0.75)", "n": int((cts_valid >= .75).sum()), "percent of complete CTS": 100*(cts_valid >= .75).mean()},
    ])
else:
    thresholds = pd.DataFrame(columns=["CTS band", "n", "percent of complete CTS"])
display(Markdown("### CTS threshold counts"))
display(thresholds.round(3))

rs_valid = model["RS"].dropna()
if len(rs_valid):
    rs_thresholds = pd.DataFrame([
        {"RS criterion": "Full RS (RS = 1)", "n": int(np.isclose(rs_valid, 1).sum()), "percent of complete RS": 100*np.isclose(rs_valid, 1).mean()},
        {"RS criterion": "RS = 0", "n": int(np.isclose(rs_valid, 0).sum()), "percent of complete RS": 100*np.isclose(rs_valid, 0).mean()},
    ])
else:
    rs_thresholds = pd.DataFrame(columns=["RS criterion", "n", "percent of complete RS"])
display(Markdown("### RS threshold counts"))
display(rs_thresholds.round(3))

cts_value_counts = cts_valid.value_counts().sort_index().rename_axis("CTS").reset_index(name="n")
cts_value_counts["percent"] = 100*cts_value_counts["n"]/len(cts_valid) if len(cts_valid) else np.nan
cts_bins = pd.cut(cts_valid, bins=np.linspace(0, 1, 11), include_lowest=True, right=True).value_counts().sort_index().rename_axis("CTS bin").reset_index(name="n")
display(Markdown("### CTS frequency table and 0.1-wide histogram bins"))
display(cts_value_counts.round(4))
display(cts_bins)

# ---------------------------------------------------------------------
# 5. Category and release-year summaries
# ---------------------------------------------------------------------
if model["Category_label"].notna().any():
    category_summary = (model.groupby("Category_label", dropna=False)
        .agg(inventory_rows=(model_col, "size"), n_CTS=("CTS", "count"), mean_CTS=("CTS", "mean"), median_CTS=("CTS", "median"), sd_CTS=("CTS", "std"),
             n_RS=("RS", "count"), mean_RS=("RS", "mean"), median_RS=("RS", "median"), sd_RS=("RS", "std"))
        .reset_index())
else:
    category_summary = pd.DataFrame(columns=["Category_label", "inventory_rows", "n_CTS", "mean_CTS", "median_CTS", "sd_CTS", "n_RS", "mean_RS", "median_RS", "sd_RS"])
display(Markdown("## Category summaries"))
display(category_summary.round(4))

with_year = model[model["Release_year_numeric"].notna()].copy()
if not with_year.empty:
    with_year["Release_year_numeric"] = with_year["Release_year_numeric"].astype(int)
    year_summary = (with_year.groupby("Release_year_numeric")
        .agg(inventory_rows=(model_col, "size"), n_CTS=("CTS", "count"), mean_CTS=("CTS", "mean"), median_CTS=("CTS", "median"), sd_CTS=("CTS", "std"),
             n_RS=("RS", "count"), mean_RS=("RS", "mean"), median_RS=("RS", "median"), sd_RS=("RS", "std"))
        .reset_index().rename(columns={"Release_year_numeric": "Release year"}))
else:
    year_summary = pd.DataFrame(columns=["Release year", "inventory_rows", "n_CTS", "mean_CTS", "median_CTS", "sd_CTS", "n_RS", "mean_RS", "median_RS", "sd_RS"])
display(Markdown("## Release-year summaries"))
display(year_summary.round(4))

try:
    from scipy.stats import spearmanr, pearsonr, kruskal
    scipy_available = True
except Exception:
    scipy_available = False
    warnings.warn("SciPy unavailable; correlation and Kruskal-Wallis tests will be skipped.")

relationship_rows = []
if scipy_available and "Training_data_depth_deficiency_proxy_5field" in model:
    paired = model[["CTS", "Training_data_depth_deficiency_proxy_5field"]].dropna()
    if len(paired) >= 3 and paired.nunique().min() > 1:
        rho, p_s = spearmanr(paired["CTS"], paired["Training_data_depth_deficiency_proxy_5field"])
        r, p_p = pearsonr(paired["CTS"], paired["Training_data_depth_deficiency_proxy_5field"])
        relationship_rows += [
            {"Relationship": "CTS vs five-field training-data deficiency proxy", "n": len(paired), "method": "Spearman", "coefficient": rho, "p_value": p_s},
            {"Relationship": "CTS vs five-field training-data deficiency proxy", "n": len(paired), "method": "Pearson", "coefficient": r, "p_value": p_p},
        ]
if scipy_available and "DDI_7component" in model:
    paired = model[["CTS", "DDI_7component"]].dropna()
    if len(paired) >= 3 and paired.nunique().min() > 1:
        rho, p_s = spearmanr(paired["CTS"], paired["DDI_7component"])
        r, p_p = pearsonr(paired["CTS"], paired["DDI_7component"])
        relationship_rows += [
            {"Relationship": "CTS vs DDI (7 component-depth fields)", "n": len(paired), "method": "Spearman", "coefficient": rho, "p_value": p_s},
            {"Relationship": "CTS vs DDI (7 component-depth fields)", "n": len(paired), "method": "Pearson", "coefficient": r, "p_value": p_p},
        ]
relationship_summary = pd.DataFrame(relationship_rows)
display(Markdown("## Score relationships"))
display(relationship_summary.round(5) if not relationship_summary.empty else Markdown("No eligible complete score pairs were available."))

category_tests = []
if scipy_available and model["Category_label"].notna().any():
    for metric in ["CTS", "RS"]:
        groups = [g[metric].dropna().to_numpy() for _, g in model.groupby("Category_label") if g[metric].notna().sum() >= 2]
        if len(groups) >= 2:
            h, p = kruskal(*groups)
            category_tests.append({"Metric": metric, "test": "Kruskal-Wallis", "groups_n": len(groups), "H": h, "p_value": p})
category_test_summary = pd.DataFrame(category_tests)
display(Markdown("## Category tests"))
display(category_test_summary.round(5) if not category_test_summary.empty else Markdown("Insufficient category-group sizes for the test."))

# ---------------------------------------------------------------------
# 6. Provenance diagnostics
# ---------------------------------------------------------------------
if "Real Audit Log Template" in excel.sheet_names:
    audit = pd.read_excel(WORKBOOK_PATH, sheet_name="Real Audit Log Template", engine="openpyxl")
    audit.columns = [str(c).strip() for c in audit.columns]
    required_aliases = {
        "source": ["URL or persistent source ID", "Source URL", "URL"],
        "access date": ["First checked date (ISO; only if documented)", "Access date", "Date accessed"],
        "snapshot": ["Archived copy / snapshot path", "Archive path", "Snapshot ID"],
        "coder": ["Accessed by / coder ID", "Coder ID", "Auditor"],
        "evidence": ["Exact evidence excerpt/location", "Evidence excerpt", "Evidence location"],
        "rationale": ["Scoring rationale / rule applied", "Scoring rationale", "Rationale"],
        "source version": ["Source version/date", "Source version", "Published date"],
    }
    logcols = {k: find_col(audit.columns, v) for k, v in required_aliases.items()}
    placeholder = re.compile(r"^(\s*|tbd|to be determined|template|placeholder|unknown|not specified|n/?a)$", re.I)
    diagnostic = []
    for field, col in logcols.items():
        if col is None:
            diagnostic.append({"Field": field, "column found": False, "nonempty rows": 0, "placeholder/blank rows": len(audit)})
        else:
            vals = audit[col].astype("string")
            filled = vals.notna() & ~vals.str.match(placeholder, na=False)
            diagnostic.append({"Field": field, "column found": True, "nonempty rows": int(filled.sum()), "placeholder/blank rows": int((~filled).sum())})
    provenance_diagnostic = pd.DataFrame(diagnostic)
else:
    audit = None
    provenance_diagnostic = pd.DataFrame({"Notice": ["No audit-log sheet found; provenance cannot be assessed from this workbook."]})
display(Markdown("## Audit-log field completeness"))
display(provenance_diagnostic)

if audit is not None:
    access_col = find_col(audit.columns, ["First checked date (ISO; only if documented)", "Access date", "Date accessed"])
    if access_col:
        access_dates = pd.to_datetime(audit[access_col], errors="coerce").dropna()
        print("\nRecorded access-date values found in the audit log:", len(access_dates))
        if len(access_dates):
            print("Earliest recorded access date:", access_dates.min().date())
            print("Latest recorded access date:", access_dates.max().date())

# ---------------------------------------------------------------------
# 7. Model-level output
# ---------------------------------------------------------------------
keep = [id_col, model_col]
if year_col: keep.append(year_col)
if category_col: keep.append(category_col)
keep += score_cols + ["CTS_components_observed", "CTS", "RS"]
if "Training_data_depth_mean_5field" in model:
    keep += ["Training_data_depth_mean_5field", "Training_data_depth_deficiency_proxy_5field"]
if "DDI_7component" in model:
    keep += ["DDI_7component"]
model_scores = model[keep].copy()
display(Markdown(f"## {DATA_LABEL}: model-row score table"))
display(model_scores.round(4))

# ---------------------------------------------------------------------
# 8. Export: CSVs + Word report
# ---------------------------------------------------------------------
if EXPORT_TABLES:
    outdir = WORKBOOK_PATH.parent / "Results_3_1_to_3_3"
    outdir.mkdir(parents=True, exist_ok=True)
    outputs = {
        "score_summary.csv": score_summary,
        "cts_thresholds.csv": thresholds,
        "rs_thresholds.csv": rs_thresholds,
        "cts_value_counts.csv": cts_value_counts,
        "cts_histogram_bins.csv": cts_bins,
        "category_summary.csv": category_summary,
        "release_year_summary.csv": year_summary,
        "score_relationships.csv": relationship_summary,
        "category_tests_exploratory.csv": category_test_summary,
        "provenance_field_diagnostic.csv": provenance_diagnostic,
        "model_row_scores.csv": model_scores,
    }
    for filename, table in outputs.items():
        table.to_csv(outdir / filename, index=False)
    print(f"\nCSV tables saved to: {outdir}")

    # ---------------- WORD REPORT ----------------
    if DOCX_AVAILABLE:
        try:
            doc = Document()
            style = doc.styles["Normal"]
            style.font.name = "Calibri"
            style.font.size = Pt(10)

            title = doc.add_heading(DATA_LABEL, level=0)
            title.alignment = WD_ALIGN_PARAGRAPH.CENTER

            p = doc.add_paragraph(); p.add_run("Workbook: ").bold = True; p.add_run(str(WORKBOOK_PATH))
            p = doc.add_paragraph(); p.add_run("Selected sheet: ").bold = True; p.add_run(str(sheet))
            p = doc.add_paragraph(); p.add_run("Random seed: ").bold = True; p.add_run(str(RANDOM_SEED))
            p = doc.add_paragraph(); p.add_run("Bootstrap reps: ").bold = True; p.add_run(str(BOOTSTRAP_REPS))

            def add_dataframe(doc, df, title_text):
                doc.add_heading(title_text, level=1)
                if df is None or len(df) == 0:
                    doc.add_paragraph("No rows available.")
                    return
                df_out = df.copy()
                for col in df_out.columns:
                    if pd.api.types.is_float_dtype(df_out[col]):
                        df_out[col] = df_out[col].map(lambda v: "" if pd.isna(v) else f"{v:.6f}")
                table = doc.add_table(rows=1, cols=len(df_out.columns))
                table.style = "Light Grid Accent 1"
                hdr = table.rows[0].cells
                for i, col in enumerate(df_out.columns):
                    hdr[i].text = str(col)
                    for run in hdr[i].paragraphs[0].runs:
                        run.bold = True
                for _, row in df_out.iterrows():
                    cells = table.add_row().cells
                    for i, col in enumerate(df_out.columns):
                        cells[i].text = str(row[col])
                doc.add_paragraph("")

            doc.add_heading("1. Inventory and Data-Quality Overview", level=1)
            ip = doc.add_paragraph()
            ip.add_run(f"Rows retained as inventory records: ").bold = True; ip.add_run(f"{len(model)}\n")
            ip.add_run(f"Distinct nonempty model-name strings: ").bold = True; ip.add_run(f"{model[model_col].nunique()}\n")
            ip.add_run(f"Distinct nonmissing IDs: ").bold = True; ip.add_run(f"{model[id_col].nunique(dropna=True)}\n")
            ip.add_run(f"Rows with complete seven-component CTS: ").bold = True; ip.add_run(f"{int(model['CTS'].notna().sum())}\n")
            ip.add_run(f"Rows with complete four-component RS: ").bold = True; ip.add_run(f"{int(model['RS'].notna().sum())}\n")
            ip.add_run(f"Rows with all seven CTS components present: ").bold = True; ip.add_run(f"{int(score_coverage.eq(7).sum())}\n")
            ip.add_run(f"Rows with one or more missing CTS components: ").bold = True; ip.add_run(f"{int(score_coverage.lt(7).sum())}\n")

            coverage_rows = []
            for key, col in cts_cols.items():
                converted = model[f"a_{key}"]
                coverage_rows.append({
                    "CTS component": key,
                    "source column": str(col),
                    "valid entries (0/1)": int(converted.notna().sum()),
                    "count of 1": int((converted == 1).sum()),
                    "count of 0": int((converted == 0).sum()),
                    "missing/invalid": int(converted.isna().sum()),
                })
            add_dataframe(doc, pd.DataFrame(coverage_rows), "2. CTS Component Coverage and Binary Distribution")

            add_dataframe(doc, score_summary, "3. Overall Score Statistics (with 95% Bootstrap CI)")
            add_dataframe(doc, thresholds, "4. CTS Threshold Counts")
            add_dataframe(doc, rs_thresholds, "5. RS Threshold Counts")
            add_dataframe(doc, cts_value_counts, "6. CTS Exact-Value Frequency Table")
            add_dataframe(doc, cts_bins, "7. CTS 0.1-Wide Histogram Bins")
            add_dataframe(doc, category_summary, "8. Category Summaries")
            add_dataframe(doc, year_summary, "9. Release-Year Summaries")
            if not relationship_summary.empty:
                add_dataframe(doc, relationship_summary, "10. Score Relationships (Spearman and Pearson)")
            if not category_test_summary.empty:
                add_dataframe(doc, category_test_summary, "11. Exploratory Category Tests (Kruskal-Wallis)")
            add_dataframe(doc, provenance_diagnostic, "12. Audit-Log Field Completeness Diagnostic")

            if audit is not None:
                access_col = find_col(audit.columns, ["First checked date (ISO; only if documented)", "Access date", "Date accessed"])
                if access_col:
                    access_dates = pd.to_datetime(audit[access_col], errors="coerce").dropna()
                    doc.add_heading("13. Recorded Access Dates (from audit log)", level=1)
                    doc.add_paragraph(f"Total recorded access-date values: {len(access_dates)}")
                    if len(access_dates):
                        doc.add_paragraph(f"Earliest recorded access date: {access_dates.min().date()}")
                        doc.add_paragraph(f"Latest recorded access date: {access_dates.max().date()}")

            add_dataframe(doc, model_scores, "14. Full Model-Row Score Table (all inventory rows)")

            doc.add_heading("15. Narrative Summary of Key Values", level=1)
            ctsrow = score_summary[score_summary["Metric"] == "CTS"]
            rsrow = score_summary[score_summary["Metric"] == "RS"]
            if not ctsrow.empty:
                r = ctsrow.iloc[0]
                doc.add_paragraph(
                    f"CTS: n = {int(r['n'])}, mean = {r['mean']:.4f}, median = {r['median']:.4f}, "
                    f"sample SD = {r['sample_SD']:.4f}, min = {r['min']:.4f}, Q1 = {r['Q1']:.4f}, "
                    f"Q3 = {r['Q3']:.4f}, max = {r['max']:.4f}, "
                    f"95% bootstrap CI for mean = [{r['mean_95CI_low']:.4f}, {r['mean_95CI_high']:.4f}]."
                )
            if len(cts_valid):
                doc.add_paragraph(f"CTS <= 0.25: {int((cts_valid <= .25).sum())} of {len(cts_valid)} ({100*(cts_valid <= .25).mean():.2f}%).")
                doc.add_paragraph(f"CTS >= 0.75: {int((cts_valid >= .75).sum())} of {len(cts_valid)} ({100*(cts_valid >= .75).mean():.2f}%).")
            if not rsrow.empty:
                r = rsrow.iloc[0]
                doc.add_paragraph(
                    f"RS: n = {int(r['n'])}, mean = {r['mean']:.4f}, median = {r['median']:.4f}, "
                    f"sample SD = {r['sample_SD']:.4f}, min = {r['min']:.4f}, max = {r['max']:.4f}, "
                    f"95% bootstrap CI for mean = [{r['mean_95CI_low']:.4f}, {r['mean_95CI_high']:.4f}]."
                )
            if len(rs_valid):
                doc.add_paragraph(f"Full RS (RS = 1): {int(np.isclose(rs_valid, 1).sum())} of {len(rs_valid)} ({100*np.isclose(rs_valid, 1).mean():.2f}%).")
                doc.add_paragraph(f"RS = 0: {int(np.isclose(rs_valid, 0).sum())} of {len(rs_valid)} ({100*np.isclose(rs_valid, 0).mean():.2f}%).")

            docx_path = outdir / "Results_3_1_to_3_3_Detailed_Report.docx"
            doc.save(docx_path)
            print(f"Detailed Word report saved to: {docx_path}")
        except Exception as e:
            print(f"[ERROR] Word report generation failed: {e}")
    else:
        print("\n[INFO] python-docx is not installed, so the Word report was skipped.")
        print("       Install it manually with:  pip install python-docx")
        print("       Then re-run this cell.")

print("\n" + "="*74)
print(DATA_LABEL)
print("="*74)

[setup] Installing missing package: python-docx
Workbook: C:\Users\rs2672\Desktop\AI Open\FinalWorkbook.xlsx
Sheets: ['READ ME', 'Model Inventory', 'Synthetic Audit Data', 'Real Audit Log Template', 'Chart Data', 'Chart Preview', 'Distribution Data']


## **LLM TRANSPARENCY AUDIT — RESULTS 3.1 to 3.3**

Selected sheet: `Synthetic Audit Data`. The notebook computes component scores, summary statistics, and threshold counts from the supplied audit data.


DEBUG: unique raw values found in each CTS source column
  code (Code present (mock)): 2 unique -> ['1', '0']
  weights (Weights present (mock)): 2 unique -> ['1', '0']
  training_data (Training data present (mock)): 2 unique -> ['1', '0']
  preprocessing (Preprocessing disclosed (mock)): 2 unique -> ['0', '1']
  evaluation (Evaluation protocol disclosed (mock)): 2 unique -> ['1', '0']
  documentation (Documentation adequate (mock)): 2 unique -> ['0', '1']
  alignment_safety (Alignment/safety disclosed (mock)): 2 unique -> ['1', '0']

DEBUG: binary value distribution per CTS component (1 / 0 / NaN)
  code: 1=58, 0=63, NaN=0
  weights: 1=76, 0=45, NaN=0
  training_data: 1=49, 0=72, NaN=0
  preprocessing: 1=43, 0=78, NaN=0
  evaluation: 1=68, 0=53, NaN=0
  documentation: 1=79, 0=42, NaN=0
  alignment_safety: 1=67, 0=54, NaN=0

Rows retained as inventory records: 121
Distinct nonempty model-name strings: 120
Distinct nonmissing IDs: 121
Rows with complete seven-component CTS: 121
Rows wi

## LLM TRANSPARENCY AUDIT — RESULTS 3.1 to 3.3: overall score statistics

,Metric,n,mean,median,sample_SD,min,Q1,Q3,max,mean_95CI_low,mean_95CI_high
0,CTS,121,0.5195,0.5714,0.2773,0.00,0.2857,0.7143,1.00,0.4711,0.5691
1,RS,121,0.5186,0.5000,0.3341,0.00,0.2500,0.7500,1.00,0.4587,0.5785
2,Training-data mean depth (5-field proxy),121,0.1955,0.0000,0.2621,0.00,0.0000,0.4500,0.95,0.1504,0.2442
3,Training-data depth-deficiency proxy (5 fields),121,0.8045,1.0000,0.2621,0.05,0.5500,1.0000,1.00,0.7583,0.8496


...

,Field,column found,nonempty rows,placeholder/blank rows
0,source,True,121,0
1,access date,True,121,0
2,snapshot,True,121,0
3,coder,True,121,0
4,evidence,True,1,120
5,rationale,True,121,0
6,source version,True,121,0



Recorded access-date values found in the audit log: 121
Earliest recorded access date: 2025-02-10
Latest recorded access date: 2025-12-18


## LLM TRANSPARENCY AUDIT — RESULTS 3.1 to 3.3: model-row score table

,List No.,Model name,Release year,Category (SYNTHETIC),a_code,a_weights,a_training_data,a_preprocessing,a_evaluation,a_documentation,a_alignment_safety,CTS_components_observed,CTS,RS,Training_data_depth_mean_5field,Training_data_depth_deficiency_proxy_5field
0,1,GPT-2,2019,Open-source,1.0,1.0,1.0,0.0,1.0,0.0,1.0,7,0.7143,1.00,0.50,0.50
1,2,Legacy ChatGPT-3.5,2022,Open-weight,0.0,1.0,1.0,0.0,1.0,1.0,1.0,7,0.7143,0.75,0.70,0.30
2,3,Default ChatGPT-3.5,2023,Open-source,1.0,1.0,1.0,0.0,1.0,1.0,1.0,7,0.8571,1.00,0.35,0.65
3,4,GPT-3.5 Turbo,2023,Open-source,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7,1.0000,1.00,0.95,0.05
4,5,ChatGPT-4,2023,Open-source,1.0,1.0,1.0,1.0,0.0,1.0,1.0,7,0.8571,0.75,0.45,0.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,117,Gemini 2.5 Pro,2025,Proprietary,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7,0.0000,0.00,0.00,1.00
117,118,GPT-o3 / GPT-03,2025,Open-source,1.0,1.0,1.0,1.0,1.0,1.0,0.0,7,0.8571,1.00,0.40,0.60
118,119,GPT-4.1,2025,Proprietary,0.0,0.0,0.0,1.0,1.0,0.0,1.0,7,0.4286,0.25,0.00,1.00
119,120,GPT-4o-mini,2025,Open-weight,1.0,1.0,1.0,0.0,1.0,1.0,1.0,7,0.8571,1.00,0.55,0.45



CSV tables saved to: C:\Users\rs2672\Desktop\AI Open\Results_3_1_to_3_3
Detailed Word report saved to: C:\Users\rs2672\Desktop\AI Open\Results_3_1_to_3_3\Results_3_1_to_3_3_Detailed_Report.docx

LLM TRANSPARENCY AUDIT — RESULTS 3.1 to 3.3
